# 13 - Classical ML Sensor Classification (Solution)

### Step 1 README: Environment Setup
This cell configures resilient imports and confirms runtime package versions for classical model training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "edge_ai_course").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise ModuleNotFoundError(
        "Could not find src/edge_ai_course. Run from inside the 'day 5 sensor' folder or a child folder."
    )

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from edge_ai_course.signal_generator import SignalConfig, CLASS_ID_TO_NAME, generate_source_signals
from edge_ai_course.signal_dataset import build_sensor_dataset, save_dataset_artifacts, assert_no_source_leakage
from edge_ai_course.signal_features import extract_features, fft_components
from edge_ai_course.signal_models import train_classical_models, train_cnn_model, convert_to_tflite
from edge_ai_course.signal_visualization import plot_signal_and_fft, plot_feature_distribution, plot_confusion_matrix, plot_comparison_bars
from edge_ai_course.model_analysis import (
    build_comparison_table,
    write_day5_report,
    estimate_classical_pipeline_memory,
    estimate_cnn_pipeline_memory,
)

print("Environment ready for Notebook 13")
print(f"Project root: {PROJECT_ROOT}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

### Step 2 README: Train Classical Models
This cell loads or builds the feature dataset, trains Logistic Regression and Random Forest models, and plots score comparisons.

In [ ]:
# Load or regenerate dataset artifacts.
features_csv = PROJECT_ROOT / "data" / "sensor" / "processed" / "sensor_features.csv"
if not features_csv.exists():
    data_bundle = build_sensor_dataset(config=SignalConfig(), mode="quick", window_size=256, overlap_fraction=0.5)
    save_dataset_artifacts(data_bundle)

features = pd.read_csv(features_csv)
feature_columns = [c for c in features.columns if c not in {"split", "source_signal_id", "window_id", "class_id", "class_name"}]

train_df = features[features["split"] == "train"]
test_df = features[features["split"] == "test"]

x_train = train_df[feature_columns].to_numpy(dtype=np.float32)
y_train = train_df["class_id"].to_numpy(dtype=np.int32)
x_test = test_df[feature_columns].to_numpy(dtype=np.float32)
y_test = test_df["class_id"].to_numpy(dtype=np.int32)

classical_results = train_classical_models(x_train, y_train, x_test, y_test)

model_names = list(classical_results.keys())
acc_values = [classical_results[name]["accuracy"] for name in model_names]
f1_values = [classical_results[name]["f1_macro"] for name in model_names]

x_pos = np.arange(len(model_names))
width = 0.35
plt.figure(figsize=(7, 3.8))
plt.bar(x_pos - width / 2, acc_values, width=width, label="Accuracy")
plt.bar(x_pos + width / 2, f1_values, width=width, label="F1 macro")
plt.xticks(x_pos, model_names, rotation=15)
plt.ylim(0, 1.05)
plt.title("Classical Model Performance")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Using features from: {features_csv}")
classical_results

### Step 3 README: Error Analysis and Interpretability
This cell displays confusion matrices and Random Forest feature importances to explain class separation.

In [ ]:
# Confusion matrices and model interpretability snippets.
class_names = [CLASS_ID_TO_NAME[i] for i in sorted(CLASS_ID_TO_NAME)]
for model_name, result in classical_results.items():
    print(f"\n=== {model_name} ===")
    print(result["classification_report"])
    plot_confusion_matrix(np.array(result["confusion_matrix"]), class_names, f"{model_name} confusion matrix")

# Random Forest importance from serialized pipeline.
import joblib
rf = joblib.load(Path(classical_results["random_forest"]["model_path"]))
rf_model = rf.named_steps["model"]
importance = pd.Series(rf_model.feature_importances_, index=feature_columns).sort_values(ascending=False).head(12)
plt.figure(figsize=(8, 4))
importance.plot(kind="bar")
plt.title("Random Forest Top Feature Importances")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()